# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook demonstrates how to load and explore the FAIR⁲ dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library.

### Dataset Source
Dataset provided as a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` is installed
!pip install mlcroissant

## 1. Data Loading
Load dataset metadata and records using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load dataset from schema
dataset = mlc.Dataset(croissant_url)

# Show available metadata fields
meta = dataset.metadata
print(f"Dataset name: {getattr(meta, 'name', '')}\n")
print(f"Description: {getattr(meta, 'description', '')}\n")
print(f"Published: {getattr(meta, 'datePublished', '')}")
print(f"Identifier: {getattr(meta, 'identifier', '')}")

## 2. Data Overview
Review available record sets, fields, and their `@id`s for subsequent extraction. The following cells enumerate the high-level structure (record sets and their fields/columns).

In [ ]:
# List available record sets in the dataset
record_sets = list(dataset.record_sets)
print("Available record sets in dataset:")
for rec in record_sets:
    print(f"- @id: {rec['@id']} | name: {rec.get('name','')}")

In [ ]:
# For each record set, list its fields (@id and name)
for rec in record_sets:
    print(f"\nFields in record set {rec['@id']}:")
    for field in rec.get('field', []):
        # A field might be a dict or a string (id)
        if isinstance(field, dict):
            field_id = field.get('@id', '')
            field_name = field.get('name', '')
        else:
            field_id = field
            field_name = ''
        print(f"   - @id: {field_id} | name: {field_name}")

In [ ]:
# For convenience, assign the main table's record set @id
# (Fill in the @id you'd like for extraction, obtained from the above cells)
# For this dataset, there's likely only one main record set.
main_record_set_id = record_sets[0]['@id'] if record_sets else None
print(f"Selected main record set: {main_record_set_id}")

## 3. Data Extraction
Load all records from the main record set as a DataFrame for downstream exploration. All entity references use their `@id` fields.

In [ ]:
dataframes = {}

if main_record_set_id:
    # Load records for the main record set
    records = list(dataset.records(record_set=main_record_set_id))
    df = pd.DataFrame(records)
    dataframes[main_record_set_id] = df
    print(f"Loaded DataFrame shape: {df.shape}")
    print("Columns:\n", df.columns.tolist())
    display(df.head())
else:
    print("No main record set detected.")

## 4. Exploratory Data Analysis (EDA)
Apply basic processing steps such as filtering, normalization, and grouping. All columns and fields referenced by their Croissant `@id`.

In [ ]:
# Choose numeric and group fields by their @id (refer to column names above)
# Substitute with the appropriate @id from your schema if the below are not present
df = dataframes.get(main_record_set_id)
example_numeric_field = None
example_group_field = None

# Automatically find a likely numeric field ('age', 'interval', etc)
numerics = df.select_dtypes('number').columns.tolist()
if numerics:
    example_numeric_field = numerics[0]
    print(f"Auto-selected numeric field: {example_numeric_field}")
else:
    print("No numeric fields found. Please specify manually.")

# Attempt to find a likely grouping field (categorical)
obj_cols = df.select_dtypes('object').columns.tolist()
if obj_cols:
    example_group_field = obj_cols[0]
    print(f"Auto-selected group field: {example_group_field}")

# Filtering example - filter where numeric field > threshold
threshold = df[example_numeric_field].quantile(0.5) if example_numeric_field else None
if example_numeric_field and threshold is not None:
    filtered_df = df[df[example_numeric_field] > threshold].copy()
    print(f"Filtered records where {example_numeric_field} > {threshold:.2f} (median): {len(filtered_df)} rows\n")
    display(filtered_df.head())
    # Normalize
    filtered_df[f"{example_numeric_field}_normalized"] = (
        filtered_df[example_numeric_field] - filtered_df[example_numeric_field].mean()
    ) / filtered_df[example_numeric_field].std()
    print(f"\nNormalized {example_numeric_field}:")
    display(filtered_df[[example_numeric_field, f"{example_numeric_field}_normalized"]].head())
    # Grouping
    if example_group_field in filtered_df.columns:
        grouped_df = (
            filtered_df.groupby(example_group_field)[example_numeric_field].mean().to_frame("mean_" + example_numeric_field)
        )
        print(f"\nGrouped mean {example_numeric_field} by {example_group_field}:")
        display(grouped_df)
else:
    print("Could not perform numeric EDA - no numeric column found.")

## 5. Visualization
Visualize distributions and relationships for key fields. Replace the field names below with Croissant `@id`s if necessary.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
if example_numeric_field:
    plt.figure(figsize=(6,4))
    sns.histplot(df[example_numeric_field], bins=10)
    plt.title(f"Distribution of {example_numeric_field}")
    plt.xlabel(example_numeric_field)
    plt.show()

if example_numeric_field and example_group_field:
    plt.figure(figsize=(8,5))
    sns.boxplot(x=example_group_field, y=example_numeric_field, data=df)
    plt.xticks(rotation=45)
    plt.title(f"{example_numeric_field} by {example_group_field}")
    plt.show()

## 6. Conclusion

- The FAIR⁲ colorectal cancer survivors dataset contains detailed clinicopathological and molecular data suitable for descriptive and biomarker prevalence analyses.
- We explored available record sets and fields by their Croissant `@id` references, loaded records into DataFrames, and performed initial cleaning and visualization.
- For more advanced modeling or extraction, refer to the detailed Croissant schema fields and use them directly by `@id` for robust pipeline development.